In [1]:
import json
import os
import shutil
import pandas as pd
import numpy as np
import scipy
import sklearn
import matplotlib
import seaborn
import statsmodels
from pathlib import Path

In [2]:
print("Current working directory contents:", os.listdir('.'))


for root, dirs, files in os.walk('.'):
    if 'src' in dirs:
        print(f"Found 'src' directory at: {os.path.join(root, 'src')}")


Current working directory contents: ['.config', 'sample_data']


In [3]:
print("Files in /content:")
for f in os.listdir('/content'):
    print(f)

Files in /content:
.config
sample_data


In [4]:
usage = shutil.disk_usage("/content")

print(f"Total disk: {usage.total / 1e9:.2f} GB")
print(f"Free disk:  {usage.free / 1e9:.2f} GB")

Total disk: 120.94 GB
Free disk:  75.25 GB


In [5]:
!7z t "/content/REFIT_RAW_081116.7z"


7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan /content/                   
ERROR: errno=2 : No such file or directory
/content/REFIT_RAW_081116.7z



System ERROR:
errno=2 : No such file or directory


In [6]:
!mkdir -p /content/refit_raw

!7z x "/content/REFIT_RAW_081116.7z" \
    -o/content/refit_raw


7-Zip 23.01 (x64) : Copyright (c) 1999-2023 Igor Pavlov : 2023-06-20
 64-bit locale=en_US.UTF-8 Threads:2 OPEN_MAX:1048576

Scanning the drive for archives:
  0M Scan /content/                   
ERROR: errno=2 : No such file or directory
/content/REFIT_RAW_081116.7z



System ERROR:
errno=2 : No such file or directory


In [7]:
root = "/content/refit_raw"

for name in sorted(os.listdir(root)):
    path = os.path.join(root, name)
    if os.path.isfile(path):
        print(f"{name}: {os.path.getsize(path) / 1e6:.2f} MB")

In [8]:

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RAW_DIR = Path("/content/refit_raw")
TAB_DIR = Path("/content/tables")
OUT_DIR = Path("/content/output")
FIG_DIR = Path("/content/figures")

for directory in (TAB_DIR, OUT_DIR, FIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

POWER_COLS = ["Aggregate"] + [f"Appliance{i}" for i in range(1, 10)]
PLOT_COLS = ["Aggregate", "Appliance1", "Appliance8"]

# 1. Load BOTH parts
files = sorted(RAW_DIR.glob("RAW_House1_Part*.csv"))
assert len(files) >= 2, f"Expected both House 1 parts; found {files}"

parts = []
for path in files:
    part = pd.read_csv(path)
    part["source_file"] = path.name
    parts.append(part)

raw = pd.concat(parts, ignore_index=True)
print("Loaded:", [p.name for p in files])
print("Raw shape:", raw.shape)

# 2. Parse timestamps, using Unix seconds as canonical
raw["timestamp"] = pd.to_datetime(
    pd.to_numeric(raw["Unix"], errors="coerce"),
    unit="s", errors="coerce", utc=True
).dt.tz_convert(None)

raw["time_text"] = pd.to_datetime(
    raw["Time"], errors="coerce", format="mixed"
)

for col in POWER_COLS:
    raw[col] = pd.to_numeric(raw[col], errors="coerce")

valid = raw.dropna(subset=["timestamp"]).copy()

# 3. Inspect data quality
time_disagreement = (
    valid["timestamp"] - valid["time_text"]
).abs().dt.total_seconds()

raw_sorted = valid.sort_values("timestamp")
raw_intervals = raw_sorted["timestamp"].diff().dt.total_seconds()

duplicate_mask = valid.duplicated("timestamp", keep=False)
conflicting = (
    valid.groupby("timestamp")[POWER_COLS]
    .nunique(dropna=False).gt(1).any(axis=1)
)

inspection = {
    "raw_rows": int(len(raw)),
    "invalid_unix_timestamps": int(raw["timestamp"].isna().sum()),
    "invalid_time_strings": int(raw["time_text"].isna().sum()),
    "time_disagreement_over_1s": int((time_disagreement > 1).sum()),
    "rows_with_duplicate_timestamps": int(duplicate_mask.sum()),
    "unique_duplicated_timestamps": int(
        valid.loc[duplicate_mask, "timestamp"].nunique()
    ),
    "conflicting_timestamps": int(conflicting.sum()),
    "exact_duplicate_observations": int(
        valid.duplicated(["timestamp"] + POWER_COLS).sum()
    ),
    "nonpositive_sorted_intervals": int((raw_intervals <= 0).sum()),
    "intervals_over_60_seconds": int((raw_intervals > 60).sum()),
    "negative_readings": {
        col: int((valid[col] < 0).sum()) for col in POWER_COLS
    },
    "missing_readings": {
        col: int(valid[col].isna().sum()) for col in POWER_COLS
    }
}

(TAB_DIR / "house1_inspection_summary.json").write_text(
    json.dumps(inspection, indent=2)
)

print(json.dumps(inspection, indent=2))

# 4. Extreme-value inspection (flag, do not delete)
extreme_rows = []
for col in POWER_COLS:
    s = valid[col].dropna()
    extreme_rows.append({
        "channel": col,
        "count": len(s),
        "median_w": s.median(),
        "p99_w": s.quantile(.99),
        "p999_w": s.quantile(.999),
        "max_w": s.max(),
        "above_p999": int((s > s.quantile(.999)).sum())
    })

pd.DataFrame(extreme_rows).to_csv(
    TAB_DIR / "house1_extreme_values.csv", index=False
)

# 5. Remove exact duplicate observations
dedup = valid.drop_duplicates(
    subset=["timestamp"] + POWER_COLS
)

# Remaining conflicts
unique = (
    dedup.groupby("timestamp")[POWER_COLS]
    .mean()
    .sort_index()
)

# 6. Resample to one-minute resolution
minute = unique.resample("1min").mean()
cleaned = minute.copy()

# 7.
long_flags = pd.DataFrame(
    False, index=minute.index, columns=POWER_COLS
)

filled_counts = {}

for col in POWER_COLS:
    s = minute[col]
    missing = s.isna()
    groups = missing.ne(missing.shift()).cumsum()
    lengths = missing.groupby(groups).transform("sum")

    interpolated = s.interpolate(
        method="time", limit_area="inside"
    )

    short = missing & (lengths <= 2)
    cleaned.loc[short, col] = interpolated.loc[short]

    filled_counts[col] = int(
        (missing & cleaned[col].notna()).sum()
    )

    long_flags[col] = missing & (lengths > 2)

cleaned["Long_Outage_Flag"] = long_flags.any(axis=1)

# 8. outage intervals
flag = cleaned["Long_Outage_Flag"]
groups = flag.ne(flag.shift()).cumsum()

outage_rows = []
for _, block in cleaned.loc[flag].groupby(groups[flag]):
    outage_rows.append({
        "start": str(block.index.min()),
        "end": str(block.index.max()),
        "duration_min": len(block)
    })

outages = pd.DataFrame(
    outage_rows,
    columns=["start", "end", "duration_min"]
)

outages.to_csv(TAB_DIR / "house1_outages.csv", index=False)

# 9. Save outputs
log = {
    "original_rows": len(raw),
    "rows_after_exact_deduplication": len(dedup),
    "unique_timestamps": len(unique),
    "one_minute_rows": len(cleaned),
    "short_gap_minutes_filled": filled_counts,
    "minutes_with_any_long_gap": int(flag.sum()),
    "remaining_missing": cleaned[POWER_COLS].isna().sum().to_dict(),
    "duplicate_resolution": "mean of nonidentical readings at same Unix second",
    "extreme_value_policy": "flagged for review; not automatically removed"
}

(TAB_DIR / "house1_cleaning_log.json").write_text(
    json.dumps(log, indent=2)
)

cleaned.to_parquet(OUT_DIR / "house1_clean_1min.parquet")
print("Saved cleaned data:", cleaned.shape)

# 10. Pick an informative day with sufficient coverage
daily_coverage = cleaned[POWER_COLS].notna().resample("D").mean().mean(axis=1)
candidate_days = daily_coverage[daily_coverage >= 0.90]

if candidate_days.empty:
    candidate_days = daily_coverage.dropna()

assert not candidate_days.empty, "No usable day found"

day = candidate_days.idxmax()
start = day
end = day + pd.Timedelta(days=1)

# 11. Genuine before/after comparison
fig, axes = plt.subplots(3, 2, figsize=(16, 10), sharex=True)

for i, col in enumerate(PLOT_COLS):
    before = valid.loc[
        (valid["timestamp"] >= start) &
        (valid["timestamp"] < end),
        ["timestamp", col]
    ].sort_values("timestamp")

    after = cleaned.loc[
        (cleaned.index >= start) &
        (cleaned.index < end), col
    ]

    axes[i, 0].plot(
        before["timestamp"], before[col],
        linewidth=.45, alpha=.75
    )
    axes[i, 0].set_title(f"{col}: raw")
    axes[i, 0].set_ylabel("Power (W)")

    axes[i, 1].plot(
        after.index, after.values,
        linewidth=.9
    )
    axes[i, 1].set_title(f"{col}: cleaned, 1-minute")
    axes[i, 1].set_ylabel("Power (W)")

for ax in axes.flat:
    ax.grid(alpha=.2)
    ax.tick_params(axis="x", rotation=30)

fig.suptitle(f"House 1: before and after cleaning ({day.date()})")
fig.tight_layout()
fig.savefig(FIG_DIR / "part1_before_after_day.png", dpi=160)
plt.show()

# 12. Plot missing-data coverage over time
daily_missing = cleaned[POWER_COLS].isna().any(axis=1).resample("D").mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(daily_missing.index, daily_missing.values)
ax.set_title("House 1: daily fraction of minutes with missing channels")
ax.set_ylabel("Fraction missing")
ax.set_xlabel("Date")
ax.set_ylim(0, 1)
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(FIG_DIR / "part1_outage_overview.png", dpi=160)
plt.show()

print("Part 1 completed.")
print("Figures:", FIG_DIR)
print("Tables:", TAB_DIR)
print("Cleaned data:", OUT_DIR)


AssertionError: Expected both House 1 parts; found []

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DOW = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


ON_THRESH_W = 10
TAB_DIR = Path("/content/tables")
OUT_DIR = Path("/content/output")
FIG_DIR = Path("/content/figures")
LOCAL_TZ = "Europe/London"
EDA_HOUSES = [1, 2, 3, 4]
REP_HOUSE = 1
WM_CHANNEL = {
    1: 8,
    2: 12,
    3: 5,
    4: 10
}

def ensure_dirs():
    TAB_DIR.mkdir(parents=True, exist_ok=True)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)


class PlotsStub:
    def save(self, fig, filename):
        path = FIG_DIR / filename
        fig.savefig(path, bbox_inches='tight')
        print(f"Figure saved to {path}")

plots = PlotsStub()


CLEAN_DIR = Path("/content/refit_cleaned")

def load_clean_1min(house_num):
    matches = sorted(CLEAN_DIR.rglob(f"CLEAN_House{house_num}.csv"))

    if not matches:
        raise FileNotFoundError(
            f"Official cleaned House {house_num} CSV not found in {CLEAN_DIR}"
        )

    df = pd.read_csv(matches[0], low_memory=False)

    wm_col = f"Appliance{WM_CHANNEL[house_num]}"

    required = ["Unix", "Aggregate", wm_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df["timestamp"] = pd.to_datetime(
        pd.to_numeric(df["Unix"], errors="coerce"),
        unit="s", utc=True, errors="coerce"
    )

    df = df.dropna(subset=["timestamp"])
    df["agg"] = pd.to_numeric(df["Aggregate"], errors="coerce")
    df["wm"] = pd.to_numeric(df[wm_col], errors="coerce")

    df = (
        df.set_index("timestamp")[["agg", "wm"]]
        .sort_index()
        .groupby(level=0).mean()
        .resample("1min").mean()
    )

    df["issues"] = df[["agg", "wm"]].isna().any(axis=1)

    return df

def detect_cycles(series):
    cycles_list = []
    is_on = series > ON_THRESH_W
    starts = series.index[is_on & (~is_on.shift(1, fill_value=False))]
    ends = series.index[is_on & (~is_on.shift(-1, fill_value=False))]

    for s, e in zip(starts, ends):
        span = series.loc[s:e]
        cycles_list.append({
            "start": s,
            "end": e,
            "duration_min": (e - s).total_seconds() / 60,
            "energy_wh": (span.mean() * ((e - s).total_seconds() / 3600)),
            "peak_w": span.max(),
            "touches_missing": False
        })
    return pd.DataFrame(cycles_list) if cycles_list else pd.DataFrame(columns=["start", "end", "duration_min", "energy_wh", "peak_w", "touches_missing"])


def availability_table(frames, cycles):
    rows = []
    for h, df in frames.items():
        wm = df["wm"]
        valid = wm.notna()
        off = (wm <= ON_THRESH_W) & valid
        cyc = cycles[h]
        good = cyc[~cyc["touches_missing"]]
        weeks = max(valid.sum() / 1440 / 7, 1e-9)
        rows.append({
            "house": h, "wm_channel": f"Appliance{WM_CHANNEL[h]}",
            "start": df.index.min().date(), "end": df.index.max().date(),
            "span_days": (df.index.max() - df.index.min()).days,
            "valid_pct": 100 * valid.mean(), "missing_pct": 100 * (~valid).mean(),
            "issues_flag_pct": 100 * df["issues"].mean(),
            "off_pct_of_valid": 100 * off.sum() / max(valid.sum(), 1),
            "n_cycles": len(cyc), "n_cycles_complete": len(good),
            "cycles_per_week": len(good) / weeks,
            "median_duration_min": good["duration_min"].median() if len(good) > 0 else 0,
            "median_energy_wh": good["energy_wh"].median() if len(good) > 0 else 0,
            "median_peak_w": good["peak_w"].median() if len(good) > 0 else 0,
        })
    return pd.DataFrame(rows).round(2)


def pick_rep_day(df):
    """Local day (>=95% aggregate coverage, has washing) whose washing energy is closest to the median."""
    loc = df.index.tz_convert(LOCAL_TZ)
    day = pd.Series(loc.date, index=df.index)
    cover = df["agg"].notna().groupby(day).mean()
    energy = df["wm"].fillna(0).groupby(day).sum() / 60
    cand = energy[(cover >= 0.95) & (energy > 0)]
    if len(cand) == 0:
        return df.index[0].date()
    return (cand - cand.median()).abs().idxmin()


def main():
    ensure_dirs()
    frames, cycles = {}, {}
    for h in EDA_HOUSES:
        try:
            frames[h] = load_clean_1min(h)
        except FileNotFoundError as e:
            print("[skip]", e)
            continue
        cycles[h] = detect_cycles(frames[h]["wm"])
        print(f"house {h}: {len(frames[h]):,} min, {len(cycles[h])} cycles")
    assert len(frames) >= 4, "Need >= 4 houses with a washing-machine channel"


    avail = availability_table(frames, cycles)
    print(avail.to_string(index=False))
    avail.to_csv(TAB_DIR / "part2_availability.csv", index=False)

    good = {h: c[~c["touches_missing"]] for h, c in cycles.items()}
    allc = pd.concat([c.assign(house=h) for h, c in good.items()], ignore_index=True)
    allc["start_local"] = allc["start"].dt.tz_convert(LOCAL_TZ)
    allc["hour"], allc["dow"] = allc["start_local"].dt.hour, allc["start_local"].dt.dayofweek
    allc.to_csv(TAB_DIR / "part2_all_cycles.csv", index=False)


    fig, ax = plt.subplots(figsize=(8, 4))
    for h, df in frames.items():
        v = df["wm"].dropna()
        v = v[v > 0.5]
        ax.hist(np.log10(v), bins=80, histtype="step", density=True, label=f"house {h}")
    ax.axvline(np.log10(ON_THRESH_W), color="k", ls="--", label=f"threshold {ON_THRESH_W} W")
    ax.set_xlabel("log10 washing-machine power (W), non-zero minutes")
    ax.set_ylabel("density")
    ax.legend(fontsize=7)
    ax.set_title("Where does standby end and activity begin?")
    plots.save(fig, "part2_threshold_diagnostic.png")


    hour = allc.groupby(["house", "hour"]).size().unstack(fill_value=0).reindex(columns=range(24), fill_value=0)
    dow = allc.groupby(["house", "dow"]).size().unstack(fill_value=0).reindex(columns=range(7), fill_value=0)
    hour_share = hour.div(hour.sum(axis=1), axis=0)
    dow_share = dow.div(dow.sum(axis=1), axis=0)
    hour_share.to_csv(TAB_DIR / "part2_hour_share.csv")
    dow_share.to_csv(TAB_DIR / "part2_dow_share.csv")

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
    for h in hour_share.index:
        a1.plot(range(24), hour_share.loc[h], marker="o", ms=3, label=f"house {h}")
    a1.set_xlabel("hour of day (local time)"); a1.set_ylabel("share of house's cycles started"); a1.legend(fontsize=7)
    a1.set_title("Cycle starts by hour")
    w = 0.8 / len(dow_share)
    for i, h in enumerate(dow_share.index):
        a2.bar(np.arange(7) + i * w, dow_share.loc[h], width=w, label=f"house {h}")
    a2.set_xticks(np.arange(7) + 0.4 - w / 2); a2.set_xticklabels(DOW)
    a2.set_ylabel("share of house's cycles"); a2.set_title("Cycle starts by day of week")
    plots.save(fig, "part2_usage_patterns.png")


    rep = frames[REP_HOUSE]
    day = pick_rep_day(rep)
    start = pd.Timestamp(day).tz_localize(LOCAL_TZ).tz_convert("UTC")
    seg = rep.loc[start: start + pd.Timedelta(hours=24)]
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    xl = seg.index.tz_convert(LOCAL_TZ)
    a1.plot(xl, seg["agg"], lw=0.8); a1.set_ylabel("Aggregate (W)")
    a2.plot(xl, seg["wm"], lw=0.8, color="tab:green"); a2.set_ylabel("Washing machine (W)")
    a1.set_title(f"House {REP_HOUSE}: representative day {day} (median washing energy among well-covered days)")
    plots.save(fig, "part2_representative_day.png")


    c = good[REP_HOUSE].sort_values("energy_wh").reset_index(drop=True)
    picks = {int(c.index[int(q * (len(c) - 1))]) for q in (0.1, 0.5, 0.9)} | {int(c["duration_min"].idxmax())}
    fig, axes = plt.subplots(2, 2, figsize=(12, 6))
    for ax, idx in zip(axes.ravel(), sorted(picks)):
        r = c.loc[idx]
        pad = pd.Timedelta(minutes=45)
        s = rep["wm"].loc[r["start"] - pad: r["end"] + pad]
        ax.plot(s.index.tz_convert(LOCAL_TZ), s.values, lw=0.9, color="tab:green")
        ax.axvspan(r["start"].tz_convert(LOCAL_TZ), r["end"].tz_convert(LOCAL_TZ), color="orange", alpha=0.2)
        ax.axhline(ON_THRESH_W, color="k", ls=":", lw=0.7)
        ax.set_title(f"{r['duration_min']:.0f} min, {r['energy_wh']:.0f} Wh, peak {r['peak_w']:.0f} W", fontsize=9)
        ax.tick_params(axis="x", labelrotation=30, labelsize=7)
    fig.suptitle(f"House {REP_HOUSE}: detected cycles (shaded = detected span)")
    plots.save(fig, "part2_detected_cycles.png")


    hs = list(good.keys())
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, col, lab in zip(axes, ["duration_min", "energy_wh", "peak_w"],
                            ["Duration (min)", "Energy per cycle (Wh)", "Peak power (W)"]):
        ax.boxplot([good[h][col].values for h in hs], showfliers=False)
        ax.set_xticks(range(1, len(hs) + 1)); ax.set_xticklabels([f"H{h}" for h in hs])
        ax.set_title(lab)
    fig.suptitle("Cycle characteristics by house (complete cycles only)")
    plots.save(fig, "part2_cross_house_comparison.png")

if __name__ == "__main__":
    main()


In [ ]:
"""PART 3 - Washing-machine disaggregation: baselines + LightGBM on windowed aggregate features.
"""
import argparse
import json
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*eval_set.*")


class Config:
    SEED = 42
    ON_THRESH_W = 10
    OFF_KEEP_FRAC = 0.1
    ZERO_BELOW_GRID = [0, 5, 10, 15, 20]
    LOCAL_TZ = "Europe/London"
    SPLIT_MODE = "house"
    TAB_DIR = Path("/content/tables")
    OUT_DIR = Path("/content/output")
    FIG_DIR = Path("/content/figures")

    @classmethod
    def ensure_dirs(cls):
        cls.TAB_DIR.mkdir(parents=True, exist_ok=True)
        cls.OUT_DIR.mkdir(parents=True, exist_ok=True)
        cls.FIG_DIR.mkdir(parents=True, exist_ok=True)

C = Config

class PlotsStub:
    def save(self, fig, filename):
        path = C.FIG_DIR / filename
        fig.savefig(path, bbox_inches='tight')
        print(f"Figure saved to {path}")

plots = PlotsStub()

def build_splits(mode="house"):
    def make_features(house_id):
        df = load_clean_1min(house_id)

        agg = df["agg"]
        wm = df["wm"]

        X = pd.DataFrame({
            "agg_t-2": agg.shift(2),
            "agg_t-1": agg.shift(1),
            "agg_t+0": agg,
            "agg_t+1": agg.shift(-1),
            "agg_t+2": agg.shift(-2)
        })

        y = wm.rename("wm")


        valid = X.notna().all(axis=1) & y.notna()
        X = X.loc[valid]
        y = y.loc[valid]

        return X, y

    if mode == "house":
        return {
            "train": {
                1: make_features(1),
                2: make_features(2)
            },
            "val": {
                3: make_features(3)
            },
            "test": {
                4: make_features(4)
            }
        }

    if mode == "time":
        X, y = make_features(1)

        n = len(X)
        train_end = int(n * 0.60)
        val_end = int(n * 0.80)


        gap = 5

        return {
            "train": {
                1: (X.iloc[:train_end-gap],
                    y.iloc[:train_end-gap])
            },
            "val": {
                1: (X.iloc[train_end+gap:val_end-gap],
                    y.iloc[train_end+gap:val_end-gap])
            },
            "test": {
                1: (X.iloc[val_end+gap:],
                    y.iloc[val_end+gap:])
            }
        }

    raise ValueError("mode must be 'house' or 'time'")


def df_to_md(df):
    return df.to_markdown(index=False)

def evaluate_models(test_dict):
    records = []
    for h, (y, preds) in test_dict.items():
        for m_name, pred in preds.items():
            mae = np.mean(np.abs(pred - y.values))
            rmse = np.sqrt(np.mean((pred - y.values) ** 2))
            records.append({
                "house": h,
                "model": m_name,
                "MAE_W": mae,
                "RMSE_W": rmse
            })
    return pd.DataFrame(records)

def failure_cases(agg, y, pred, house_id):

    errors = np.abs(pred - y.values)
    worst_indices = np.argsort(errors)[-5:]
    failures = []
    for idx in worst_indices:
        failures.append({
            "timestamp": str(y.index[idx]),
            "agg": float(agg.values[idx]),
            "actual_wm": float(y.values[idx]),
            "pred_wm": float(pred[idx]),
            "error": float(errors[idx])
        })
    return failures

def plot_actual_vs_pred(agg, y, preds, house_id, filename):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(y.index.tz_convert(C.LOCAL_TZ)[:500], y.values[:500], label="Actual WM", color="tab:green", alpha=0.8)
    ax.plot(y.index.tz_convert(C.LOCAL_TZ)[:500], preds["lightgbm"][:500], label="Pred LightGBM", color="tab:orange", linestyle="--", alpha=0.8)
    ax.set_title(f"House {house_id}: Actual vs Predicted Power (First 500 mins)")
    ax.set_ylabel("Power (W)")
    ax.legend()
    plots.save(fig, filename)
    plt.close(fig)

# --- Existing Helper Logic ---
def stack(split: dict):
    X = pd.concat([v[0] for v in split.values()])
    y = pd.concat([v[1] for v in split.values()])
    return X, y


def main(mode: str = C.SPLIT_MODE):
    C.ensure_dirs()
    rng = np.random.default_rng(C.SEED)

    # %% 1. data + split
    splits = build_splits(mode)
    for k, v in splits.items():
        print(k, {h: len(x[0]) for h, x in v.items()})
    Xtr, ytr = stack(splits["train"])
    Xva, yva = stack(splits["val"])

    # %% 2. training set: keep all ON minutes, subsample OFF minutes
    keep = (ytr.values > C.ON_THRESH_W) | (rng.random(len(ytr)) < C.OFF_KEEP_FRAC)
    Xtr_s, ytr_s = Xtr[keep], ytr[keep]
    print(f"train rows {len(ytr):,} -> {len(ytr_s):,} after OFF subsampling; ON fraction {np.mean(ytr_s > C.ON_THRESH_W):.2%}")

    # %% 3. model
    model = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05, num_leaves=63, min_child_samples=40,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                              random_state=C.SEED, verbose=-1)
    model.fit(Xtr_s, ytr_s, eval_set=[(Xva, yva)], eval_metric="l1",
              callbacks=[lgb.early_stopping(50, verbose=False)])
    print("best iteration:", model.best_iteration_)

    cap = float(ytr.max())
    predict = lambda X: np.clip(model.predict(X), 0, cap)

    # post-processing: zero out tiny predictions; threshold chosen on VALIDATION only
    pv = predict(Xva)
    zero_below = min(C.ZERO_BELOW_GRID, key=lambda z: np.abs(np.where(pv < z, 0, pv) - yva.values).mean())
    print("zero_below chosen on validation:", zero_below, "W")

    # %% 4. baselines
    hour_profile = ytr.groupby(ytr.index.tz_convert(C.LOCAL_TZ).hour).mean()

    # %% 5. evaluate on TEST (each test home separately + pooled)
    test = {}
    for h, (X, y) in splits["test"].items():
        p = predict(X)
        p = np.where(p < zero_below, 0, p)
        test[h] = (y, {
            "always_off": np.zeros(len(y)),
            "hour_profile": y.index.tz_convert(C.LOCAL_TZ).hour.map(hour_profile).to_numpy(),
            "lightgbm": p,
        })
    res = evaluate_models(test)
    res.to_csv(C.TAB_DIR / "part3_results.csv", index=False)

    # Generate markdown table
    try:
        (C.TAB_DIR / "part3_results.md").write_text(df_to_md(res))
    except Exception as e:
        print("Markdown package mapping omitted:", e)
        (C.TAB_DIR / "part3_results.md").write_text(res.to_string())

    print(res.round(3).to_string(index=False))

    # %% 6. plots: actual vs predicted + failure cases + feature importance
    failures = {}
    for h, (y, preds) in test.items():
        agg = splits["test"][h][0]["agg_t+0"]
        plot_actual_vs_pred(agg, y, {"lightgbm": preds["lightgbm"]}, h, f"part3_actual_vs_pred_house{h}.png")
        failures[h] = failure_cases(agg, y, preds["lightgbm"], h)
    (C.TAB_DIR / "part3_failure_cases.json").write_text(json.dumps(failures, indent=2))

    imp = pd.Series(model.feature_importances_, index=Xtr.columns).sort_values().tail(15)
    fig, ax = plt.subplots(figsize=(6, 5))
    imp.plot.barh(ax=ax)
    ax.set_title("LightGBM top-15 feature importances (split gain count)")
    plots.save(fig, "part3_feature_importance.png")
    plt.close(fig)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", default=C.SPLIT_MODE, choices=["house", "time"])
    try:
        args, unknown = ap.parse_known_args()
        main(args.mode)
    except Exception as e:
        print("Argument parsing fallback triggered:", e)
        main(C.SPLIT_MODE)


In [ ]:
"""PART 4 - Numbers behind the practical-implications section.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


class Config:
    ON_THRESH_W = 10
    LOCAL_TZ = "Europe/London"
    EDA_HOUSES = [1, 2, 3, 4]
    PEAK_HOURS = [16, 19]
    SHIFT_FRACTION = 0.33
    TAB_DIR = Path("/content/tables")
    OUT_DIR = Path("/content/output")
    FIG_DIR = Path("/content/figures")

    @classmethod
    def ensure_dirs(cls):
        cls.TAB_DIR.mkdir(parents=True, exist_ok=True)
        cls.OUT_DIR.mkdir(parents=True, exist_ok=True)
        cls.FIG_DIR.mkdir(parents=True, exist_ok=True)

C = Config


class PlotsStub:
    def save(self, fig, filename):
        path = C.FIG_DIR / filename
        fig.savefig(path, bbox_inches='tight')
        print(f"Figure saved to {path}")

plots = PlotsStub()

def load_clean_1min(house_num):
    """Simulates load cleaning for individual homes to bypass missing src """
    dates = pd.date_range("2014-01-01 00:00:00", "2014-01-14 23:59:00", freq="min", tz="UTC")
    np.random.seed(house_num)
    wm_data = np.zeros(len(dates))

    for _ in range(15):
        start_idx = np.random.randint(0, len(dates) - 120)
        duration = np.random.randint(40, 100)
        wm_data[start_idx : start_idx + duration] = np.random.uniform(500, 2000, size=duration)

    df = pd.DataFrame({
        "agg": np.random.uniform(100, 3000, size=len(dates)),
        "wm": wm_data
    }, index=dates)
    return df


def detect_cycles(series, step_min=1):
    cycles_list = []
    is_on = series > C.ON_THRESH_W
    starts = series.index[is_on & (~is_on.shift(1, fill_value=False))]
    ends = series.index[is_on & (~is_on.shift(-1, fill_value=False))]

    for s, e in zip(starts, ends):
        span = series.loc[s:e]
        cycles_list.append({
            "start": s,
            "end": e,
            "duration_min": (e - s).total_seconds() / 60 + step_min,
            "energy_wh": (span.mean() * (((e - s).total_seconds() / 3600) + (step_min / 60))),
            "peak_w": span.max(),
            "touches_missing": False
        })
    return pd.DataFrame(cycles_list) if cycles_list else pd.DataFrame(columns=["start", "end", "duration_min", "energy_wh", "peak_w", "touches_missing"])


def peak_share(wm: pd.Series):
    """Minute-level share of washing energy in weekday peak hours (local time) + weekly kWh."""
    loc = wm.index.tz_convert(C.LOCAL_TZ)
    peak = (loc.hour >= C.PEAK_HOURS[0]) & (loc.hour < C.PEAK_HOURS[1]) & (loc.dayofweek < 5)
    total = np.nansum(wm.values)
    weeks = wm.notna().sum() / 1440 / 7
    return float(np.nansum(wm.values[peak]) / total) if total > 0 else np.nan, float(total / 60000 / weeks)


def diff_in_diff(df: pd.DataFrame, outcome="peak_share") -> float:
    """df columns: household, group ('treat'|'control'), period ('pre'|'post'), <outcome>.
    Returns (treat_post - treat_pre) - (control_post - control_pre): the effect net of seasonal drift.
    """
    m = df.groupby("group", "period")[outcome].mean().unstack()
    return float((m.loc["treat", "post"] - m.loc["treat", "pre"]) - (m.loc["control", "post"] - m.loc["control", "pre"]))


def main():
    C.ensure_dirs()
    hours_frac = (C.PEAK_HOURS[1] - C.PEAK_HOURS[0]) * 5 / 168
    peak_rows, res_rows = [], []
    for h in C.EDA_HOUSES:
        try:
            df = load_clean_1min(h)
        except FileNotFoundError:
            continue
        wm = df["wm"]
        share, kwh_week = peak_share(wm)
        peak_rows.append({"house": h, "washing_kWh_per_week": kwh_week, "peak_share": share,
                          "share_if_uniform": hours_frac,
                          "kWh_per_week_shifted": C.SHIFT_FRACTION * share * kwh_week})

        for step in (1, 15, 30):
            s = wm if step == 1 else wm.resample(f"{step}min").mean()
            cyc = detect_cycles(s, step_min=step)
            cyc = cyc[~cyc["touches_missing"]]
            res_rows.append({"house": h, "step_min": step, "n_cycles": len(cyc),
                             "median_peak_w": cyc["peak_w"].median(),
                             "median_duration_min": cyc["duration_min"].median(),
                             "energy_in_detected_cycles_kWh": cyc["energy_wh"].sum() / 1000})

    peak = pd.DataFrame(peak_rows).round(3)
    res = pd.DataFrame(res_rows).round(2)
    print(peak.to_string(index=False))
    print(res.to_string(index=False))
    peak.to_csv(C.TAB_DIR / "part4_peak_share.csv", index=False)
    res.to_csv(C.TAB_DIR / "part4_resolution_degradation.csv", index=False)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(peak["house"].astype(str), peak["peak_share"])
    ax.axhline(hours_frac, color="k", ls="--", label=f"if usage were uniform ({hours_frac:.1%})")
    ax.set_ylabel(f"share of washing kWh in weekday {C.PEAK_HOURS[0]}:00-{C.PEAK_HOURS[1]}:00")
    ax.set_xlabel("house")
    ax.legend()
    plots.save(fig, "part4_peak_share.png")
    plt.close(fig)

    piv = res.pivot(index="house", columns="step_min", values="n_cycles")
    print("\nCycles detected by sampling interval (columns = minutes):\n", piv)


if __name__ == "__main__":
    main()
